# Redes Neurais Convolucionais com PyTorch

## Introdução

### Estrutura de Pastas

Confira abaixo a estrutura de pastas de um projeto

```shell
├── data                         # Repositório de imagens
├── src
│   ├── deep_learning
│   │   ├── data                 # Leitura e preparo das imagens
│   │   │   ├── __init__.py
│   │   │   ├── dataset.py       # Classe que lê as imagens do disco
│   │   │   └── transforms.py    # Pipelines de pré-processamento e aumento de dados
│   │   ├── engine               # Motor de treinamento
│   │   │   ├── __init__.py
│   │   │   ├── predictor.py     # Função de inferência para classificar fotos
│   │   │   └── trainer.py       # Loops de treino, validação e early stopping
│   │   ├── models               # Modelos de redes neurais
│   │   │   ├── __init__.py
│   │   │   ├── ...
│   │   │   └── ...
│   │   ├── config.py
│   │   └── __init__.py
│   └── __init__.py
├── main.py                      # Script fino que conecta tudo
├── pyproject.toml
└── uv.lock
```

## Hardware e Dispositivo

Ao treinar e utilizar redes neurais, podemos utilizar tanto o processador como a placa de vídeo para operar os cálculos com tensores 4d. De longe, a placa de vídeo é o dispositivo mais adequado, mas nem sempre está disponível. No PyTorch podemos configurar o dispositivo utilizando o método device. O método device deverá receber na maioria do tempo um de dois valores: "cpu" para processador ou "cuda" para placa de vídeo. Para saber se a placa de vídeo pode ser utilizada, chamamos o método is_available de torch.cuda. O método retornara True caso a gpu esteja disponível. Geralmente, selecionamos o dispositivo conforme o exemplo abaixo. Assim, a gpu é sempre usada quando possível.

In [2]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Dispositivo selecionado: {device}")

Dispositivo selecionado: cuda


### Tensores Fictícios

Para fazer algumas operações com pytorch, utilizamos tensores que representam imagens fictícias. Para gerar um tensor fictício, podemos utilizar o método randn do pytorch. O método receberá como parâmetros o tamanho de cada dimensão do tensor. No exemplo abaixo, estamos gerando um tensor 4d no formato (B, C, H, W) contento 16 imagens fictícias coloridas de 224x224.

IMPORTANTE: Sempre que criarmos um tensor no python, o padrão do pytorch é criá-lo no processador, utilizando a memória RAM do sistema. Quando encadeamos o método to, passando como parâmetro o dispositivo, estamos transferindo o tensor para o dispositivo. Nesse exemplo, o tensor nasce no processador e logo depois o transferimos para a VRAM (Memória RAM da GPU).

In [4]:
x = torch.randn(16, 3, 224, 224).to(device)

Em um problema de classificação, precisaremos tanto das imagens quanto do gabarito. Podemos simular um gabarito utilizando o método randint do pytorch. O método randint gerará um tensor de números inteiros. No exemplo abaixo, utilizamos o parâmetro low para definir o valor mínimo, high para definir o valor máximo e size para definir as dimensões do tensor. Nesse caso, uma única dimensão de tamanho 16.

Ao lidar com tensores, geralmente estaremos em uma dentre duas situações: gerando um tensor do zero ou manipulando um tensor já existente. Quando criamos um tensor do zero, podemos definir o dispositivo diretamente na chamada do método utilizando parâmetro device, conforme o exemplo abaixo. No nosso contexto, o tensor de gabarito nascerá diretamente na memória da gpu (VRAM). Quando o tensor já existe e nós não o geramos, só podemos transferí-lo entre as memórias. Então o tensor nasceu no processador e terá de ser transferida para a gpu, conforme necessário. Para tensores minúsculos como estes que estamos gerando, a diferença é negligenciável. Mas para matrizes enormes que podem superar gigabytes de tamanho, o ideal é sempre gerá-las direto na gpu para evitar gargalos desnecessários da memória RAM.

No exemplo, o que estamos fazendo é, em resumo: gerando um tensor com 16 números aleatórios de 0 à 3 diretamente na memória de vídeo (VRAM).

In [5]:
y = torch.randint(low=0, high=3, size=(16, ), device=device)

Eis o resultado. Abaixo, utilizamos a propriedade shape dos tensores para inspecionar o formato deles. Como queríamos, o tensor fictício de imagens tem quatro dimensões com os tamanhos que especificamos e o tensor fictício do gabarito tem uma dimensão de tamanho 16.

In [6]:
print(x.shape)
print(y.shape)

torch.Size([16, 3, 224, 224])
torch.Size([16])


## Rede Sequencial

Nessa segunda etapa, definimos uma rede sequencial utilizando a classe Sequential de torch.nn. Essa classe receberá vários passos em ordem. O primeiro passo que definimos foi um passo de convolução com a classe Conv2d de nn. Essa classe recebe alguns parâmetros como in_channels, que define a quantidade de canais do input. O parâmetro out_channels define a quantidade de canais do output. O parâmetro kernel_size define o tamanho da lente de convolução e stride define o passo. O que vai acontecer nesse primeiro passo é que será aplicada uma convolução para um tensor de 3 canais (RGB) calculando o produto escalar a partir de uma matriz 3x3 ao redor do pixel atual, pulando os pixels de um a um. Em uma convolução, quando utilizamos um kernel 3x3, teremos como resultado uma imagem com 2 pixels a menos por dimensão do que a imagem original. Como nossas imagens são de 224x224, esta convolução retornaria naturalmente 222x222. Para preservar o tamanho da imagem, especificamos um padding que adicionará uma quantidade de pixels nas bordas. Assim, a imagem seguirá preservando o mesmo tamanho. Aqui, entregamos um tensor 4d no formato (16, 16, 224, 224).

Após aplicar uma convolução, utilizamos a classe ReLU de nn. Que trará a não-linearidade para o modelo. Senão, a rede neural seria apenas uma função linear absurdamente grande. O parâmetro inplace define se os bytes serão modificados no mesmo espaço de memória. Se sim, não será necessário alocar memória RAM extra. Seja a memória do processador ou da gpu. Aqui, entregamos um tensor 4d no formato (16, 16, 224, 224).

A classe MaxPool2d dividirá a imagem em janelas, selecionando o maior valor de cada janela. Nesse exemplo, definimos kernel_size como 2. Então as janelas serão de 2x2. Também definimos stride (passo) como 2. Então pularemos de 2 em 2 pixels. Na prática, essa etapa entregará de volta uma imagem de 112x112 contendo apenas os pixels de maior valor da imagem original. Aqui, entregamos um tensor 4d no formato (16, 16, 112, 112).

Tendo as imagens de 112x112 em mãos, passamos elas para uma classe AdaptiveAvgPool2d. Essa classe recebe um parâmetro output_size que define o tamanho das dimensões da resposta. Como é 2d, tem de ser duas dimensões. No exemplo, definimos que a dimensão deve ser 1x1. Então entra 112x112 e sai 1x1. Essa classe calculará uma média de todos os 112x112 pixels da imagem e entregará de volta essa média em tamanho 1x1 (um único valor). Aqui, entregamos um tensor 4d no formato (16, 16, 1, 1).

Em sequência, a classe Flatten receberá o tensor anterior e o transformará em uma matriz (tensor 2d), mantendo o lote intacto. Aqui, entregamos um tensor 2d no formato (16, 16).

Por fim, a classe Linear realiza uma combinação linear dos atributos extraídos. Essa classe receberá um parâmetro in_features que indica o número de entradas que cada imagem entrega. Este parâmetro tem valor condizente com a quantidade de canais da imagem (nesse caso, 16). O parâmetro out_features determina a quantidade de saídas por amostra. Nesse caso, são três saídas, já que temos três classes de classificação. 

In [8]:
import torch.nn as nn

model = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.AdaptiveAvgPool2d(output_size=(1, 1)),
    nn.Flatten(),
    nn.Linear(in_features=16, out_features=3)
).to(device)

Agora, vamos tentar trazer toda a teoria para o código. O coração desse modelo é o primeiro passo: a convolução. No primeiro ciclo, o pytorch atribuirá pesos completamente aleatórios para cada pixel das lentes de convolução. Após isso, virão vários passos de transformação como a função ReLU para garantir a não linearidade e manter apenas o que ativou a lente de convolução. As transformações continuam até chegar ao classificador (nn.Linear). 

Em problemas de classificação, geralmente temos de resumir todos os números dos pixels gerados pelas lentes de convolução a poucos números. É exatamente isso que fizemos usando a classe AdaptiveAvgPool2d, que resumiu cada canal entregue pelas lentes de convolução a um único número. Formando assim um conjunto de 16 números para os 16 canais. Mas, por quê resumir a poucos números? Porque em problemas de classificação, a posição do objetos não interessa. Se gerássemos matrizes maiores como 7x7, ainda preservaríamos alguma noção de posição, o que não é importante para classificar uma imagem. Além disso, imagine o seguinte. A última camada da rede neural terá de assimilar todos os valores que a camada linear recebeu. Então se tivermos matrizes 7x7, serão 49 valores para cada canal. Com 16 canais, ainda é aceitável, totalizando 7 x 7 x 16 = 784 números. Mas, no mundo real, sempre utilizaremos modelos pré-prontos que não utilizam apenas 16 canais, e sim 256 ou 512 canais por exemplo. Assim, a quantidade de números na última camada da rede neural atingirá a proporção de dezenas de milhares de números. O restante da rede neural precisará escalar para atingir esse patamar, criando milhões de parâmetros a mais. Isso torna a rede muito mais pesada e muito mais propensa a decorar a imagem (overfitting).

### Formato da Saída

Para gerar as predições, podemos chamar a função model que é retornada pela classe Sequential. Passando como parâmetro as imagens. Isso devolverá um tensor 2d no qual a primeira camada tem o tamanho de acordo com o número de imagens do lote e a segunda camada contém a quantidade de features que definimos para a saída. Nesse caso, três features.

In [9]:
predictions = model(x)

print(f"Formato de saída: {predictions.shape}")

Formato de saída: torch.Size([16, 3])


Os dados que recebemos como saída são o que chamamos de logits. Os logits são números brutos desconexos que podem variar do infinito negativo ao infinito positivo. Cada classe de classificação terá seu respectivo logit. Como o resultado das predições é uma matriz de formato (16, 3), cada item da primeira camada é o conjunto de pontuações de uma imagem do input. Para cara conjunto de 3 números, o primeiro logit é a pontuação da classe 0. O segundo logit é a pontuação da classe 1 e o terceiro logit é a pontuação da classe 2. Para que esses números tenham algum significado, é preciso passá-los por uma equação de conversão.

In [10]:
print(f"Dados de saída: \n{predictions}\n")

Dados de saída: 
tensor([[ 0.6205, -0.0739,  0.4677],
        [ 0.6208, -0.0696,  0.4693],
        [ 0.6220, -0.0733,  0.4659],
        [ 0.6174, -0.0757,  0.4701],
        [ 0.6159, -0.0738,  0.4685],
        [ 0.6173, -0.0736,  0.4692],
        [ 0.6177, -0.0721,  0.4663],
        [ 0.6193, -0.0741,  0.4702],
        [ 0.6181, -0.0755,  0.4720],
        [ 0.6212, -0.0710,  0.4673],
        [ 0.6169, -0.0740,  0.4676],
        [ 0.6190, -0.0758,  0.4729],
        [ 0.6136, -0.0745,  0.4700],
        [ 0.6195, -0.0743,  0.4658],
        [ 0.6171, -0.0735,  0.4708],
        [ 0.6197, -0.0757,  0.4698]], device='cuda:0',
       grad_fn=<AddmmBackward0>)



## Calculando o Erro

A equação que utilizaremos para dar sentido aos logits é a equação softmax. Em resumo, ela converte aqueles números desconexos para porcentagens. A soma das porcentagens de cada resultado de imagem tem de atingir exatamente 100%. Dessa forma, conseguimos dizer qual classe o modelo acha mais provável de ser a correta.

In [12]:
probabilities = torch.softmax(predictions, dim=1)

print(probabilities)

tensor([[0.4241, 0.2118, 0.3641],
        [0.4236, 0.2124, 0.3640],
        [0.4247, 0.2119, 0.3633],
        [0.4232, 0.2116, 0.3652],
        [0.4229, 0.2122, 0.3649],
        [0.4231, 0.2120, 0.3649],
        [0.4235, 0.2125, 0.3640],
        [0.4235, 0.2117, 0.3648],
        [0.4230, 0.2114, 0.3655],
        [0.4241, 0.2123, 0.3636],
        [0.4233, 0.2121, 0.3646],
        [0.4231, 0.2112, 0.3656],
        [0.4222, 0.2122, 0.3657],
        [0.4242, 0.2120, 0.3638],
        [0.4228, 0.2119, 0.3653],
        [0.4238, 0.2114, 0.3648]], device='cuda:0', grad_fn=<SoftmaxBackward0>)


Na prática, não aplicaremos manualmente a equação softmax. Utilizaremos uma função de perda. Para isso, utilizamos a classe CrossEntropyLoss de nn. É exatamente essa classe que aplicará a equação de conversão nos logits de resultado.

Se lembra que ao completar um ciclo de treinamento, devemos calcular a direção para a qual os pesos teriam de ir para aumentar o erro e então ajustá-los na direção contrária para diminuir o erro? Além de aplicar a equação de conversão, essa classe também é a responsável por calcular a direção de aumento do erro.

In [13]:
criterion = nn.CrossEntropyLoss()

## Otimização

Agora que as predições de um ciclo já foram entregues e a direção para aumento do erro já está calculada, é hora de ajustar os pesos na direção contrária a fim de diminuir o erro.

A class Adam de torch.optim será a responsável exatamente por essa tarefa. Como primeiro parâmetro, ela receberá os parâmetros do modelo que são exatamente os pesos e os viéses. Também podemos especificar um parâmetro lr (learning rate) que definirá a intensidade do passo que o algoritmo de otimização dará na direção contrária ao erro. Passos muito longos pulam o ponto de erro mínimo. Passos muito curtos demoram excessivamente para convergir. O valor padrão do Adam é de 0.001. 

In [15]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Conectando os Pontos

Finalmente, vamos colocar as coisas para funcionar de fato. Primeiro, precisamos informar à rede neural que ela deve entrar em fase de treinamento. Para isso, chamamos o método train do modelo.

Após isso, faremos um laço de repetição. Esse laço repetirá a sequência de todo um ciclo de treinamento. Os ciclos de treinamento são o que chamamos de épocas. Abaixo, definimos que o treinamento ocorrerá em 10 épocas. 

Para cada época, chamamos o método zero_grad do otimizador. Se não fizermos essa chamada, os gradientes dos lotes anteriores se somarão aos novos indefinidamente, até que em um momento, os pesos da rede virem NaN (Não Número). 

Após zerar os gradientes, realizamos as predições e calculamos a perda utilizando a função criterion que geramos a partir de CrossEntropyLoss. Para calcular o erro, passamos como parâmetros os resultados da predição e o gabarito. O resultado do erro será um tensor com um único número que representa a média daquele erro do lote.

Após calcular o erro, chamamos o método backward do tensor de erro. Assim, o motor de diferenciação automática percorre a rede de trás para frente, calculando as derivadas parciais de cada peso da rede. 

Por fim, o método step do otimizador irá ler os resultados das derivadas parciais e aplicar os ajustes nos parâmetros da rede neural.

In [16]:
model.train()

for epoch in range(10):
    optimizer.zero_grad()

    predictions = model(x)
    loss = criterion(predictions, y)

    loss.backward()
    optimizer.step()

    print(f"Época: {epoch + 1}, Perda: {loss.item():.4f}")

Época: 1, Perda: 1.1845
Época: 2, Perda: 1.1764
Época: 3, Perda: 1.1685
Época: 4, Perda: 1.1610
Época: 5, Perda: 1.1538
Época: 6, Perda: 1.1469
Época: 7, Perda: 1.1404
Época: 8, Perda: 1.1342
Época: 9, Perda: 1.1283
Época: 10, Perda: 1.1227
